In [1]:
# pyautogen>=0.4 renamed the module to autogen_agentchat — pin to <0.4 for
# the classic ConversableAgent API used in this notebook.
# litellm is required by CrewAI to route to Groq (or any non-native provider).
# crewai checks for litellm at IMPORT TIME — if crewai was already imported
# before litellm was installed, you must restart the kernel for the check to pass.
%pip install -q "pyautogen<0.4" tavily-python crewai crewai-tools litellm python-dotenv

print("")
print("*** IMPORTANT: Restart the kernel now before running any other cell. ***")
print("    Kernel -> Restart Kernel  (or the restart button in the toolbar)")
print("    Then run all cells from the top.")
print("    Reason: crewai caches the litellm availability check at import time.")



*** IMPORTANT: Restart the kernel now before running any other cell. ***
    Kernel -> Restart Kernel  (or the restart button in the toolbar)
    Then run all cells from the top.
    Reason: crewai caches the litellm availability check at import time.


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()  # loads from .env file if present

# Set your API keys here if not using a .env file
os.environ["GROQ_API_KEY"] = "REMOVED_KEY"
os.environ["TAVILY_API_KEY"] = "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"

GROQ_API_KEY   = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

print(f"GROQ_API_KEY:   {'set' if GROQ_API_KEY else 'NOT SET — add to .env'}")
print(f"TAVILY_API_KEY: {'set' if TAVILY_API_KEY else 'NOT SET — add to .env'}")

GROQ_API_KEY:   set
TAVILY_API_KEY: set


In [3]:
!pip uninstall -y numpy
!pip install numpy==1.26.4

!pip install -U autogen flaml

# optional but safer
!pip install ray

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 7.9 MB/s eta 0:00:00


In [4]:
from autogen import ConversableAgent
from tavily import TavilyClient

# ── LLM Configuration ──────────────────────────────────────────────────────
# Note: AutoGen 0.3.x sends tools using the old OpenAI 'functions' schema,
# which Groq rejects with a 400 error regardless of model.
# Solution: we handle the web search in plain Python (Tavily), inject the
# results into the researcher's message, then let AutoGen handle the
# multi-agent conversation (synthesise → write → translate).
# This is cleaner and teaches the same multi-agent concepts.
groq_config = [
    {
        "model": "llama-3.1-8b-instant", # Changed model here for faster execution
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",
        "api_type": "openai",
    }
]

llm_config = {"config_list": groq_config, "temperature": 0.3, "max_tokens": 800} # Added max_tokens for consistency
print("LLM config ready — using Groq / llama-3.1-8b-instant") # Updated print statement

LLM config ready — using Groq / llama-3.1-8b-instant


In [5]:
# ── Tool: Web Search via Tavily ─────────────────────────────────────────────
def search_internet(query: str) -> str:
    """
    Search the web using the Tavily API.

    Args:
        query (str): The search query string.

    Returns:
        str: Search results as a formatted string.
    """
    client = TavilyClient(api_key=TAVILY_API_KEY)
    try:
        response = client.search(query=query, max_results=5)
        # Format results cleanly
        results = []
        for r in response.get("results", []):
            results.append(f"Title: {r.get('title', '')}\nURL: {r.get('url', '')}\nContent: {r.get('content', '')}")
        return "\n\n---\n\n".join(results)
    except Exception as e:
        return f"Search error: {str(e)}"

print("search_internet tool defined.")

search_internet tool defined.


In [6]:
# ── Plain Python web search (Tavily) ────────────────────────────────────────
# We call Tavily directly in Python instead of through AutoGen's tool system.
# AutoGen 0.3.x uses the deprecated 'functions' API which Groq no longer accepts.
def run_web_search(query: str, max_results: int = 5) -> str:
    """Run a Tavily web search and return formatted results."""
    client = TavilyClient(api_key=TAVILY_API_KEY)
    try:
        response = client.search(query=query, max_results=max_results)
        results = []
        for r in response.get("results", []):
            results.append(
                f"Title: {r.get('title', '')}\n"
                f"URL: {r.get('url', '')}\n"
                f"Content: {r.get('content', '')}"
            )
        return "\n\n---\n\n".join(results)
    except Exception as e:
        return f"Search error: {str(e)}"

print("Web search function ready.")

# ── Agent 1: User Proxy ─────────────────────────────────────────────────────
# The UserProxy acts as the human — no LLM, terminates on 'TERMINATE'.
user_proxy = ConversableAgent(
    name="UserProxy",
    llm_config=False,
    human_input_mode="NEVER",
    code_execution_config=False,
    is_termination_msg=lambda x: "TERMINATE" in (x.get("content") or ""),
    default_auto_reply="Continue if not finished, otherwise say TERMINATE."
)

# ── Agent 2: Researcher ─────────────────────────────────────────────────────
# Receives pre-fetched search results and synthesises them into a report.
researcher_agent = ConversableAgent(
    name="Researcher",
    llm_config=llm_config,
    system_message=(
        "You are a professional research analyst. "
        "You will be given raw web search results on a topic. "
        "Synthesise them into a structured research report with: key facts, "
        "important statistics, key developments, and source URLs. "
        "When your report is complete, end your final message with 'TERMINATE'."
    )
)

# ── Agent 3: Writer ─────────────────────────────────────────────────────────
writer_agent = ConversableAgent(
    name="Writer",
    llm_config=llm_config,
    system_message=(
        "You are a professional content writer. "
        "Take research findings and transform them into a well-structured, "
        "engaging article with: an introduction, 3-4 clearly titled sections, "
        "key takeaways, and proper source citations. "
        "When your article is complete, end with 'TERMINATE'."
    )
)

# ── Agent 4: Translator ─────────────────────────────────────────────────────
translator_agent = ConversableAgent(
    name="Translator",
    llm_config=llm_config,
    system_message=(
        "You are a professional translator. "
        "Translate the given article into Spanish, preserving the original "
        "meaning, tone, structure, and all section headings. "
        "When the translation is complete, end with 'TERMINATE'."
    )
)

print("All agents created.")

Web search function ready.
All agents created.


In [7]:
# ── Step 1: Search (Python) then Research (AutoGen) ────────────────────────
RESEARCH_TOPIC = "Recent advances in large language models in 2024 and 2025"

# Step 1a — Run the actual web search in plain Python
print(f"Searching the web for: '{RESEARCH_TOPIC}'...")
raw_search_results = run_web_search(RESEARCH_TOPIC, max_results=5)
print(f"Search complete. Got {len(raw_search_results)} characters of results.")

# Step 1b — Hand results to the Researcher agent to synthesise into a report
# The agent receives the raw results and uses its LLM to analyse + structure them.
print(f"\nResearcher agent synthesising report...")
print("=" * 60)

research_result = user_proxy.initiate_chat(
    researcher_agent,
    message=(
        f"Topic: {RESEARCH_TOPIC}\n\n"
        f"Here are the raw web search results:\n\n{raw_search_results}\n\n"
        "Please synthesise these into a structured research report with key facts, "
        "important statistics, and source URLs."
    ),
    max_turns=4
)

research_findings = research_result.chat_history[-1]["content"].replace("TERMINATE", "").strip()
print("\nResearch complete.")

Searching the web for: 'Recent advances in large language models in 2024 and 2025'...
Search complete. Got 1557 characters of results.

Researcher agent synthesising report...
UserProxy (to Researcher):

Topic: Recent advances in large language models in 2024 and 2025

Here are the raw web search results:

Title: Large Language Models That Will Redefine AI in 2025 - GrowExx
URL: https://www.growexx.com/blog/large-language-models-that-will-redefine-ai-in-2025/
Content: Explore how large language models will revolutionize AI in 2025, advancing automation, creativity, and transforming business applications.

---

Title: Top Large Language Models of 2025 | Best LLMs Compared - Nurix AI
URL: https://www.nurix.ai/blogs/which-llm-most-advanced-today
Content: In 2025, large language models (LLMs) are powering everything from AI agents to real-time analytics. With rapid advancements in reasoning,

---

Title: Advances in Large Language Model Training: Insights from ICLR ...
URL: https://www.pap

Researcher (to UserProxy):

**Research Report: Recent Advances in Large Language Models in 2024 and 2025**

**Executive Summary:**
Large language models (LLMs) have made significant advancements in 2024 and 2025, revolutionizing AI applications in automation, creativity, and business transformation. This report summarizes key facts, important statistics, and trends in LLM development, highlighting the potential impact of these models on various industries.

**Key Facts:**

1. **Rapid Advancements:** LLMs have made rapid progress in 2024 and 2025, with significant improvements in reasoning, natural language understanding, and generation capabilities. (Source: Nurix AI)
2. **Business Applications:** LLMs are being used to power AI agents, real-time analytics, and other business applications, transforming industries such as healthcare, finance, and customer service. (Source: Nurix AI)
3. **ICLR 2025 Publications:** Recent ICLR 2025 publications have advanced our understanding of LLM train

In [8]:
# ── Step 2: Write ───────────────────────────────────────────────────────────
print("Starting article writing...")
print("=" * 60)

writing_result = user_proxy.initiate_chat(
    writer_agent,
    message=f"Based on the following research findings, write a well-structured article:\n\n{research_findings}",
    max_turns=4
)

article = writing_result.chat_history[-1]["content"].replace("TERMINATE", "").strip()
print("\nArticle writing complete.")

Starting article writing...
UserProxy (to Writer):

Based on the following research findings, write a well-structured article:

**Research Report: Recent Advances in Large Language Models in 2024 and 2025**

**Executive Summary:**
Large language models (LLMs) have made significant advancements in 2024 and 2025, revolutionizing AI applications in automation, creativity, and business transformation. This report summarizes key facts, important statistics, and trends in LLM development, highlighting the potential impact of these models on various industries.

**Key Facts:**

1. **Rapid Advancements:** LLMs have made rapid progress in 2024 and 2025, with significant improvements in reasoning, natural language understanding, and generation capabilities. (Source: Nurix AI)
2. **Business Applications:** LLMs are being used to power AI agents, real-time analytics, and other business applications, transforming industries such as healthcare, finance, and customer service. (Source: Nurix AI)
3. **

Writer (to UserProxy):

**Revolutionizing AI: Recent Advances in Large Language Models**

In recent years, large language models (LLMs) have undergone significant transformations, revolutionizing the field of artificial intelligence (AI) and transforming various industries. These models have made rapid progress in 2024 and 2025, with substantial improvements in reasoning, natural language understanding, and generation capabilities. In this article, we will delve into the recent advancements in LLMs, highlighting their potential impact on different sectors and industries.

**Section 1: Rapid Advancements in LLMs**

LLMs have made rapid progress in 2024 and 2025, with significant improvements in reasoning, natural language understanding, and generation capabilities (Nurix AI, 2025). These advancements have enabled LLMs to perform complex tasks, such as logical deduction and problem-solving, with increased accuracy and efficiency (Nurix AI, 2025). The rapid progress of LLMs has been drive

Writer (to UserProxy):

**Year in Review and 2025 Trends.** Retrieved from https://www.psychologytoday.com/us/blog/the-future-brain/202501/large-language-models-2024-year-in-review-and-2025-trends
4. GrowExx. (2025). Large language models that will redefine AI in 2025. Retrieved from https://www.growexx.com/blog/large-language-models-that-will-redefine-ai-in-2025/

**Conclusion:**
In conclusion, large language models have made significant advancements in 2024 and 2025, revolutionizing AI applications and transforming various industries. As these models continue to evolve, we can expect to see even more innovative applications and breakthroughs in the field. However, it is essential to address the challenges associated with the development and deployment of LLMs to ensure their safe and responsible use.

**Future Directions:**
The future of LLMs holds immense promise, with potential applications in areas such as:

1. **Healthcare:** LLMs can be used to analyze medical data, diagnose dis

In [9]:
# ── Step 3: Translate ───────────────────────────────────────────────────────
print("Starting translation to Spanish...")
print("=" * 60)

translation_result = user_proxy.initiate_chat(
    translator_agent,
    message=f"Please translate the following article into Spanish:\n\n{article}",
    max_turns=4
)

translated_article = translation_result.chat_history[-1]["content"].replace("TERMINATE", "").strip()
print("\nTranslation complete.")

Starting translation to Spanish...
UserProxy (to Translator):

Please translate the following article into Spanish:

**Year in Review and 2025 Trends.** Retrieved from https://www.psychologytoday.com/us/blog/the-future-brain/202501/large-language-models-2024-year-in-review-and-2025-trends
4. GrowExx. (2025). Large language models that will redefine AI in 2025. Retrieved from https://www.growexx.com/blog/large-language-models-that-will-redefine-ai-in-2025/

**Conclusion:**
In conclusion, large language models have made significant advancements in 2024 and 2025, revolutionizing AI applications and transforming various industries. As these models continue to evolve, we can expect to see even more innovative applications and breakthroughs in the field. However, it is essential to address the challenges associated with the development and deployment of LLMs to ensure their safe and responsible use.

**Future Directions:**
The future of LLMs holds immense promise, with potential applications

Translator (to UserProxy):

**Resumen del Año y Tendencias de 2025.** Recuperado de https://www.psychologytoday.com/us/blog/the-future-brain/202501/large-language-models-2024-year-in-review-and-2025-trends
4. GrowExx. (2025). Grandes modelos de lenguaje que redefinirán la IA en 2025. Recuperado de https://www.growexx.com/blog/large-language-models-that-will-redefine-ai-in-2025/

**Conclusión:**
En conclusión, los grandes modelos de lenguaje han logrado avances significativos en 2024 y 2025, revolucionando las aplicaciones de la inteligencia artificial y transformando diversas industrias. A medida que estos modelos continúan evolucionando, podemos esperar ver aún más aplicaciones innovadoras y avances en el campo. Sin embargo, es esencial abordar los desafíos asociados con el desarrollo y la implementación de LLMs para garantizar su uso seguro y responsable.

**Direcciones Futuras:**
El futuro de los LLMs tiene una gran promesa, con posibles aplicaciones en áreas como:

1. **Salud:** Lo

Translator (to UserProxy):

**Grandes modelos de lenguaje que redefinirán la IA en 2025.** Recuperado de https://www.growexx.com/blog/large-language-models-that-will-redefine-ai-in-2025/

**Conclusión:**
En conclusión, los grandes modelos de lenguaje han logrado avances significativos en 2024 y 2025, revolucionando las aplicaciones de la inteligencia artificial y transformando diversas industrias. A medida que estos modelos continúan evolucionando, podemos esperar ver aún más aplicaciones innovadoras y avances en el campo. Sin embargo, es esencial abordar los desafíos asociados con el desarrollo y la implementación de LLMs para garantizar su uso seguro y responsable.

**Direcciones Futuras:**
El futuro de los LLMs tiene una gran promesa, con posibles aplicaciones en áreas como:

1. **Salud:** Los LLMs pueden utilizarse para analizar datos médicos, diagnosticar enfermedades y desarrollar planes de tratamiento personalizados.
2. **Educación:** Los LLMs pueden utilizarse para crear sistem

In [10]:
# ── Final Output ────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("PIPELINE COMPLETE — Final Outputs")
print("=" * 60)

print("\n--- RESEARCH FINDINGS (excerpt) ---")
print(research_findings[:800] + "..." if len(research_findings) > 800 else research_findings)

print("\n--- ARTICLE (excerpt) ---")
print(article[:800] + "..." if len(article) > 800 else article)

print("\n--- TRANSLATED ARTICLE (excerpt) ---")
print(translated_article[:800] + "..." if len(translated_article) > 800 else translated_article)


PIPELINE COMPLETE — Final Outputs

--- RESEARCH FINDINGS (excerpt) ---
**Research Report: Recent Advances in Large Language Models in 2024 and 2025**

**Executive Summary:**
Large language models (LLMs) have made significant advancements in 2024 and 2025, revolutionizing AI applications in automation, creativity, and business transformation. This report summarizes key facts, important statistics, and trends in LLM development, highlighting the potential impact of these models on various industries.

**Key Facts:**

1. **Rapid Advancements:** LLMs have made rapid progress in 2024 and 2025, with significant improvements in reasoning, natural language understanding, and generation capabilities. (Source: Nurix AI)
2. **Business Applications:** LLMs are being used to power AI agents, real-time analytics, and other business applications, transforming industries suc...

--- ARTICLE (excerpt) ---
**Year in Review and 2025 Trends.** Retrieved from https://www.psychologytoday.com/us/blog/the-fu

In [11]:
import os
from crewai import Agent, Task, Crew, LLM
from crewai.tools import tool
from tavily import TavilyClient
import time

# litellm reads GROQ_API_KEY from environment — set it explicitly here
# so it works even if the .env file was not loaded before crewai imported.
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# llama-3.1-8b-instant: 560 t/s, ~1M TPM on free tier (vs 12K for 70b).
# max_tokens=800 caps per-response size to stay well under rate limits.
crew_llm = LLM(
    model="groq/llama-3.1-8b-instant",
    temperature=0.3,
    max_tokens=800,
    api_key=GROQ_API_KEY
)

print("CrewAI LLM configured — groq/llama-3.1-8b-instant")


CrewAI LLM configured — groq/llama-3.1-8b-instant


In [12]:
# ── Tool: Web Search (CrewAI style with @tool decorator) ────────────────────
@tool("Web Search Tool")
def crew_search_internet(query: str) -> str:
    """
    Search the web for the given query using the Tavily API.
    Always pass a clear, specific search query string as input.

    Args:
        query (str): The search query.

    Returns:
        str: Web search results including titles, URLs, and content snippets.
    """
    client = TavilyClient(api_key=TAVILY_API_KEY)
    try:
        response = client.search(query=query, max_results=5)
        results = []
        for r in response.get("results", []):
            results.append(f"Title: {r.get('title', '')}\nURL: {r.get('url', '')}\nContent: {r.get('content', '')}")
        return "\n\n---\n\n".join(results)
    except Exception as e:
        return f"Search error: {str(e)}"

print("CrewAI search tool defined.")

CrewAI search tool defined.


In [13]:
# Agents with short backstories to minimise prompt token usage.
researcher = Agent(
    role="Research Analyst",
    goal="Find key facts using web search and return exactly 5 bullet points with source URLs.",
    backstory="Expert researcher who produces brief bullet-point summaries.",
    tools=[crew_search_internet],
    verbose=False,
    llm=crew_llm
)
writer = Agent(
    role="Content Writer",
    goal="Turn bullet points into a 150-word article with 4 paragraphs.",
    backstory="Concise tech writer who produces short, structured articles.",
    verbose=False,
    llm=crew_llm
)
translator = Agent(
    role="Translator",
    goal="Translate the given English text into Spanish accurately.",
    backstory="Professional English-Spanish translator.",
    verbose=False,
    llm=crew_llm
)
print("Agents defined.")


Agents defined.


In [14]:
TOPIC = "Recent advances in large language models in 2024 and 2025"

research_task = Task(
    description=(
        f"Search for: {TOPIC}. "
        "Return ONLY 5 bullet points. Each bullet: one key fact + source URL. "
        "Each bullet must be under 25 words. No intro, no conclusion."
    ),
    agent=researcher,
    expected_output="5 bullet points, each under 25 words, with source URL."
)
writing_task = Task(
    description=(
        "Using the 5 research bullets, write a SHORT article: "
        "intro (1 sentence), 2 body paragraphs (2 sentences each), conclusion (1 sentence). "
        "Total: 80-100 words maximum."
    ),
    agent=writer,
    context=[research_task],
    expected_output="An 80-100 word article, 4 paragraphs."
)
translation_task = Task(
    description="Translate the article into Spanish. Output the translation only.",
    agent=translator,
    context=[writing_task],
    expected_output="The article translated into Spanish."
)
print("Tasks defined.")


Tasks defined.


In [15]:
import time

def run_with_retry(crew_obj, max_attempts=5):
    for attempt in range(1, max_attempts + 1):
        try:
            return crew_obj.kickoff()
        except Exception as e:
            err = str(e)
            if "rate_limit" in err.lower() or "429" in err or "RateLimitError" in err:
                wait = attempt * 30
                print(f"  Rate limit hit (attempt {attempt}/{max_attempts}). Sleeping {wait}s...")
                time.sleep(wait)
            else:
                raise
    print("All retries exhausted. Wait a minute then re-run this cell.")
    return None

crew = Crew(
    agents=[researcher, writer, translator],
    tasks=[research_task, writing_task, translation_task],
    verbose=False
)
print("Starting crew...")
crew_result = run_with_retry(crew)
if crew_result:
    print("Done.")


Starting crew...


  Rate limit hit (attempt 1/5). Sleeping 30s...


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'agent_execution_error' closed 'llm_call_started' (expected 
'agent_execution_started')

[CrewAIEventsBus] Warning: Event pairing mismatch. 'task_failed' closed 'agent_execution_started' (expected 
'task_started')

...

  Rate limit hit (attempt 4/5). Sleeping 120s...


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

  Rate limit hit (attempt 5/5). Sleeping 150s...

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

All retries exhausted. Wait a minute then re-run this cell.


In [16]:
if crew_result is None:
    print("No result — rate limit retries exhausted. Re-run the kickoff cell.")
else:
    print("\n" + "=" * 60)
    print("CREWAI RESULTS")
    print("=" * 60)
    print("\n--- RESEARCH REPORT ---")
    print(str(crew_result.tasks_output[0])[:800])
    print("\n--- ARTICLE ---")
    print(str(crew_result.tasks_output[1])[:800])
    print("\n--- SPANISH TRANSLATION ---")
    print(str(crew_result.tasks_output[2])[:800])

No result — rate limit retries exhausted. Re-run the kickoff cell.


In [17]:
import os
from crewai import Agent, Task, Crew, LLM
from textwrap import dedent
import time

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# llama-3.1-8b-instant: faster and higher rate limits than 70b.
# max_tokens=800 caps code output size to avoid TPM exhaustion.
devyan_llm = LLM(
    model="groq/llama-3.1-8b-instant",
    temperature=0.2,
    max_tokens=800,
    api_key=GROQ_API_KEY
)

print("Devyan LLM configured — groq/llama-3.1-8b-instant")


Devyan LLM configured — groq/llama-3.1-8b-instant


In [18]:
# Short backstories to minimise token usage.
architect = Agent(
    role="Software Architect",
    goal="Output a function signature, parameter descriptions, return type, and 3 edge cases. Max 100 words.",
    backstory="Architect who writes concise design notes.",
    allow_delegation=False, verbose=False, llm=devyan_llm
)
programmer = Agent(
    role="Python Developer",
    goal="Implement the function as a single clean Python code block with type hints and docstring.",
    backstory="Python developer who writes concise, working code.",
    allow_delegation=False, verbose=False, llm=devyan_llm
)
tester = Agent(
    role="QA Engineer",
    goal="Write exactly 5 pytest test functions as a single code block. No explanations.",
    backstory="QA engineer who writes concise targeted tests.",
    allow_delegation=False, verbose=False, llm=devyan_llm
)
reviewer = Agent(
    role="Code Reviewer",
    goal="Output 3 feedback bullets and one short usage paragraph. Max 80 words total.",
    backstory="Tech lead who writes brief, actionable reviews.",
    allow_delegation=False, verbose=False, llm=devyan_llm
)
print("Devyan agents defined.")


Devyan agents defined.


In [19]:
from crewai import Task
USER_PROBLEM = (
    "Build process_students(records: list[dict], min_score: float) -> dict that "
    "filters by min_score, sorts descending, returns {filtered, average, top_student}."
)

architecture_task = Task(
    description=(
        f"Design: {USER_PROBLEM} "
        "Output: function signature with type hints, param descriptions, return type, "
        "3 edge cases. Maximum 100 words."
    ),
    agent=architect,
    expected_output="Function signature, params, return, 3 edge cases. Under 100 words."
)
implementation_task = Task(
    description=(
        "Implement process_students() per the architecture. "
        "Output ONLY a Python code block. Type hints, docstring, edge cases. No prose."
    ),
    agent=programmer,
    context=[architecture_task],
    expected_output="A Python code block only."
)
testing_task = Task(
    description=(
        "Write exactly 5 pytest test functions for process_students(). "
        "Output ONLY the test code block. No prose, no explanations."
    ),
    agent=tester,
    context=[implementation_task],
    expected_output="Exactly 5 pytest functions as a code block."
)
reviewing_task = Task(
    description=(
        "Review the code and tests. Output ONLY: "
        "3 bullet points of feedback, then 1 sentence on how to run it. "
        "Max 80 words total."
    ),
    agent=reviewer,
    context=[architecture_task, implementation_task, testing_task],
    expected_output="3 bullets + 1 sentence. Under 80 words."
)
print("Devyan tasks defined.")


Devyan tasks defined.


In [20]:
devyan_crew = Crew(
    agents=[architect, programmer, tester, reviewer],
    tasks=[architecture_task, implementation_task, testing_task, reviewing_task],
    verbose=False
)
print("Devyan starting...")
print("Problem:", USER_PROBLEM)
print("=" * 60)
devyan_result = run_with_retry(devyan_crew)
if devyan_result:
    print("Devyan run complete.")


Devyan starting...
Problem: Build process_students(records: list[dict], min_score: float) -> dict that filters by min_score, sorts descending, returns {filtered, average, top_student}.
Devyan run complete.


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [21]:
if devyan_result is None:
    print("No result — rate limit retries exhausted. Re-run the kickoff cell.")
else:
    labels = ["ARCHITECTURE", "IMPLEMENTATION", "TEST SUITE", "REVIEW & DOCS"]
    print("\n" + "=" * 60)
    print("DEVYAN — FINAL DELIVERABLES")
    print("=" * 60)
    for i, (label, output) in enumerate(zip(labels, devyan_result.tasks_output)):
        print(f"\n{chr(45)*60}")
        print(f"[{i+1}] {label}")
        print(f"{chr(45)*60}")
        print(str(output))


DEVYAN — FINAL DELIVERABLES

------------------------------------------------------------
[1] ARCHITECTURE
------------------------------------------------------------
### Function Signature
```python
def process_students(records: list[dict], min_score: float) -> dict:
```

### Parameters
- `records`: A list of dictionaries containing student records.
- `min_score`: The minimum score to filter student records.

### Return Type
- A dictionary containing the filtered student records, average score, and top student.

### Return Value
```python
{
    'filtered': list[dict],
    'average': float,
    'top_student': dict
}
```

### Edge Cases
1. **Empty Records**: `process_students([], 0.0)` returns `{'filtered': [], 'average': 0.0, 'top_student': None}`.
2. **No Students with Min Score**: `process_students([{'name': 'John', 'score': 80}], 90.0)` returns `{'filtered': [], 'average': 0.0, 'top_student': None}`.
3. **Single Student with Min Score**: `process_students([{'name': 'John', 'score'